In [1]:
import os
import random
import numpy as np
import torch
import application
import config
import importlib
from IPython.display import clear_output
importlib.reload(application)
importlib.reload(config)

# Which dataset?
# AU dataset is named after the Dispatchable Unit ID (DUID) of the power plant
# DUID = 'EDENVSF1'#'EDENVSF1' 'RUGBYR1' 'BANN1'
# datat = DUID + '_old'
# datae = DUID + '_new'

# BE
dataset = 'Elia'
# day-ahead forecast
# datat = f'{dataset}_2016-2018_1h_dah'
# datae = f'{dataset}_2019_1h_dah'
# hour-ahead forecast
datat = f'{dataset}_2016-2018_1h_hah'
datae = f'{dataset}_2019_1h_hah'

experiment_title = f"seeded-experiment_{datat}"

# BESS capacity scenarios (in pu of max installed capacity)
bcap_scenarios = [0.5, 0.4, 0.3, 0.2, 0.1]

# How many experiments per scenario?
number_of_runs = 1

# W&B tracking
os.environ["WANDB_MODE"] = "offline"
os.environ['WANDB_API_KEY'] = "WANDB_API_KEY"
os.environ['WANDB_SILENT'] = "true"
os.environ['WANDB_RUN_GROUP'] = experiment_title

# archive the results to this path:
target_path = os.path.join('..', 'results', f'{experiment_title}.zip')

In [2]:
# Function to seed everything
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Function to log seeds
def log_seed(runid: str, seed: int, log_file: str):
  print(f"{runid=}, {seed=}, {log_file=}")
  with open(log_file, "a") as f:
    f.write(f"{runid}, {seed}\n")

In [ ]:
bess_properties = config.bess_properties

for bcap in bcap_scenarios:
  bess_properties['energy_capacity_puh'] = bcap
  random_seeds = [random.randint(0, 2**32 - 1) for _ in range(number_of_runs)]
  for i, seed in enumerate(random_seeds):
    print(f'======= b{bcap} Run No. {i+1} of {number_of_runs} =======')
    seed_everything(seed)
    wandb_run_id = str(i)
    wandb_run_id = application.train_ddpg_tracked_by_wandb(bcap=bcap, dataset=datat)
    log_seed(wandb_run_id, seed, "runid_seed_log.csv")
    csv_path = application.eval_ddpg(run_id=wandb_run_id, dataset=datae, bess_properties=bess_properties)
    clear_output() # clear verbos output to prevent page freezing; wandb keeps the log.

# archive the results
!zip  -qr {target_path} models/ evaluation/ *.csv

In [ ]:
# purge models/ evaluation/ *.csv before next run.
!rm -r models/ evaluation/ *.csv